# MLSM Lab 1 — Entrenamiento interactivo

Ejecuta las celdas en orden. Cada interacción con el usuario está separada en una celda: selección del dataset, selección del modelo y selección de la fila para la matriz de confusión.

El notebook utiliza `model_definitions.py`, que contiene los pipelines y grids de parámetros específicos de cada dataset.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists() and (BASE_DIR / "Lab1" / "data").exists():
    BASE_DIR = BASE_DIR / "Lab1"
DATA_DIR = BASE_DIR / "data"

DATASETS = {
    "1": ("GenreAll", "GenreAll.csv", "genre"),
    "2": ("features-30-sec", "features_30_sec.csv", "label"),
    "3": ("features-3-sec", "features_3_sec.csv", "label"),
}
MODEL_OPTIONS = {"l": "Logistic Regression", "k": "k-NN", "d": "Decision Tree", "a": "Artificial Neural Network"}

## Definiciones de modelos

Esta celda define los pipelines y grids de parámetros específicos de cada dataset. Así, el notebook puede ejecutarse sin importar código desde otro archivo.

In [ ]:
"""Model definitions for the MLSM Lab 1 datasets.

Each dataset has its own pipelines and parameter grids so they can be tuned
independently.
"""

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


def _genreall_models(n_features):
    """Models for GenreAll, including variance and feature selection."""
    selector_options = [
        k for k in [80, 100, 130, 150, 180, 200] if k <= n_features
    ]
    selector_options.append("all")

    common = [
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold(threshold=0.0)),
    ]

    return {
        "Logistic Regression": (
            Pipeline(common + [
                ("scaler", StandardScaler()),
                ("selector", SelectKBest(score_func=f_classif)),
                ("model", LogisticRegression(max_iter=5000, solver="lbfgs")),
            ]),
            {
                "selector__k": selector_options,
                "model__C": [0.03, 0.1, 0.2, 0.3, 0.5, 1.0],
            },
        ),
        "k-NN": (
            Pipeline(common + [
                ("scaler", StandardScaler()),
                ("selector", SelectKBest(score_func=f_classif)),
                ("model", KNeighborsClassifier()),
            ]),
            {
                "selector__k": selector_options,
                "model__n_neighbors": [5, 7, 9, 11, 15],
                "model__weights": ["uniform", "distance"],
                "model__p": [1, 2],
            },
        ),
        "Decision Tree": (
            Pipeline(common + [
                ("selector", SelectKBest(score_func=f_classif)),
                ("model", DecisionTreeClassifier(random_state=42)),
            ]),
            {
                "selector__k": selector_options,
                "model__max_depth": [5, 10, 15],
                "model__min_samples_leaf": [5, 10],
                "model__min_samples_split": [2, 10],
            },
        ),
        "Artificial Neural Network": (
            Pipeline(common + [
                ("scaler", StandardScaler()),
                ("selector", SelectKBest(score_func=f_classif)),
                ("model", MLPClassifier(
                    solver="adam",
                    learning_rate_init=0.001,
                    max_iter=700,
                    random_state=42,
                    early_stopping=True,
                    validation_fraction=0.1,
                    n_iter_no_change=15,
                    tol=1e-3,
                )),
            ]),
            {
                "selector__k": selector_options,
                "model__hidden_layer_sizes": [(50,), (100,), (100, 50)],
                "model__alpha": [0.001, 0.01, 0.1],
            },
        ),
    }


def _features_30_sec_models():
    """Models and parameter grids specifically for the 30-second dataset."""
    return {
        "Logistic Regression": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=5000, solver="lbfgs")),
            ]),
            {"model__C": [0.01, 0.03, 0.1, 0.3, 1.0]},
        ),
        "k-NN": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", KNeighborsClassifier()),
            ]),
            {
                "model__n_neighbors": [3, 5, 7, 9, 13, 17],
                "model__weights": ["uniform", "distance"],
                "model__p": [1, 2],
            },
        ),
        "Decision Tree": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DecisionTreeClassifier(random_state=42)),
            ]),
            {
                "model__max_depth": [5, 10, 15],
                "model__min_samples_split": [5, 10],
                "model__min_samples_leaf": [5, 10],
            },
        ),
        "Artificial Neural Network": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", MLPClassifier(
                    max_iter=500,
                    random_state=42,
                    early_stopping=False,
                    n_iter_no_change=15,
                    tol=1e-3,
                )),
            ]),
            {
                "model__hidden_layer_sizes": [(30,), (50,), (100,), (50, 25)],
                "model__alpha": [0.01, 0.1, 1.0],
            },
        ),
    }


def _features_3_sec_models():
    """Models and parameter grids specifically for the 3-second dataset.

    This dataset has many more samples than the 30-second dataset, so the
    grids are deliberately different and kept moderate to control runtime.
    """
    return {
        "Logistic Regression": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=5000, solver="lbfgs")),
            ]),
            {"model__C": [0.03, 0.1, 0.3, 1.0, 3.0]},
        ),
        "k-NN": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", KNeighborsClassifier()),
            ]),
            {
                "model__n_neighbors": [5, 9, 15, 25],
                "model__weights": ["distance"],
                "model__p": [1, 2],
            },
        ),
        "Decision Tree": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DecisionTreeClassifier(random_state=42)),
            ]),
            {
                "model__max_depth": [10, 20, None],
                "model__min_samples_split": [5, 10, 20],
                "model__min_samples_leaf": [2, 5, 10],
            },
        ),
        "Artificial Neural Network": (
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", MLPClassifier(
                    solver="adam",
                    learning_rate_init=0.001,
                    max_iter=500,
                    random_state=42,
                    early_stopping=True,
                    validation_fraction=0.1,
                    n_iter_no_change=15,
                    tol=1e-3,
                )),
            ]),
            {
                "model__hidden_layer_sizes": [(50,), (100,), (100, 50)],
                "model__alpha": [0.001, 0.01, 0.1],
            },
        ),
    }


def get_models(dataset, n_features):
    """Return the model pipelines and parameter grids for one dataset."""
    if dataset == "GenreAll":
        return _genreall_models(n_features)
    if dataset == "features-30-sec":
        return _features_30_sec_models()
    if dataset == "features-3-sec":
        return _features_3_sec_models()
    raise ValueError(f"Unknown dataset: {dataset}")


## 1. Seleccionar dataset

In [ ]:
print("Selecciona el dataset:\n  1 -> GenreAll\n  2 -> features-30-sec\n  3 -> features-3-sec")
dataset_choice = input("Dataset [1/2/3]: ").strip()
if dataset_choice not in DATASETS:
    raise ValueError("Opción no válida. Vuelve a ejecutar esta celda e introduce 1, 2 o 3.")
dataset_name, csv_filename, target_column = DATASETS[dataset_choice]
print(f"Dataset seleccionado: {dataset_name}")

## 2. Cargar y preparar los datos

In [ ]:
csv_path = DATA_DIR / csv_filename
if not csv_path.exists():
    raise FileNotFoundError(f"No se encuentra {csv_path}. Comprueba que los CSV están en Lab1/data/.")

df = pd.read_csv(csv_path, skipinitialspace=True).dropna(subset=[target_column])
y = df[target_column].copy()
X = df.drop(columns=[target_column]).copy()
for column in X.select_dtypes(include=["object", "string"]).columns:
    values = X[column].dropna().unique()
    if len(values) == 2:
        X[column] = X[column].map({values[0]: 0, values[1]: 1})
X = X.select_dtypes(include=np.number)
if X.empty:
    raise ValueError(f"No se han encontrado características numéricas en {csv_filename}.")
print(f"Archivo: {csv_filename}\nMuestras: {len(df)}\nCaracterísticas numéricas: {X.shape[1]}")
display(y.value_counts().rename("Número de muestras por clase").to_frame())

## 3. Seleccionar modelo

In [ ]:
available_models = get_models(dataset_name, X.shape[1])
print("Selecciona el modelo:\n  l -> Logistic Regression\n  k -> k-NN\n  d -> Decision Tree\n  a -> Artificial Neural Network\n  all -> Todos")
model_choice = input("Modelo [l/k/d/a/all]: ").strip().lower()
if model_choice == "all":
    selected_models = available_models.copy()
elif model_choice in MODEL_OPTIONS:
    model_name = MODEL_OPTIONS[model_choice]
    if model_name not in available_models:
        raise ValueError(f"{model_name} no está definido para {dataset_name}.")
    selected_models = {model_name: available_models[model_name]}
else:
    raise ValueError("Opción no válida. Vuelve a ejecutar la celda e introduce l, k, d, a o all.")
print("Modelos seleccionados:", list(selected_models))

## 4. Configurar validación cruzada

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
print("Configuración: StratifiedKFold de 10 particiones, shuffle=True, random_state=42")

## 5. Entrenar y evaluar configuraciones

Esta celda puede tardar varios minutos, especialmente si se selecciona `all`.

In [ ]:
results = []
for name, (pipeline, param_grid) in selected_models.items():
    print(f"Entrenando: {name}")
    search = GridSearchCV(estimator=pipeline, param_grid=param_grid, scoring="accuracy", cv=cv, return_train_score=True, n_jobs=-1, error_score="raise")
    search.fit(X, y)
    for i, params in enumerate(search.cv_results_["params"]):
        train_accuracy = search.cv_results_["mean_train_score"][i]
        cv_accuracy = search.cv_results_["mean_test_score"][i]
        results.append({"Algorithm": name, "Parameters": params, "Train accuracy": train_accuracy, "CV accuracy": cv_accuracy, "Overfitting gap": train_accuracy - cv_accuracy})
    print(f"Mejores parámetros: {search.best_params_}")
    print(f"Mejor CV accuracy: {search.best_score_:.4f}")

results_df = pd.DataFrame(results).sort_values(["CV accuracy", "Overfitting gap"], ascending=[False, True]).reset_index(drop=True)
print(f"Total de configuraciones evaluadas: {len(results_df)}")

## 6. Mostrar tabla de resultados

In [ ]:
display(results_df.style.format({"Train accuracy": "{:.4f}", "CV accuracy": "{:.4f}", "Overfitting gap": "{:.4f}"}))

## 7. Elegir fila para la matriz de confusión

Introduce el índice de la fila que aparece en la tabla, o `q` para omitir la matriz.

In [ ]:
row_choice = input(f"Fila de resultados [0-{len(results_df) - 1}] o q: ").strip().lower()
if row_choice == "q":
    selected_row_index = None
else:
    try:
        selected_row_index = int(row_choice)
    except ValueError as exc:
        raise ValueError("Introduce un índice válido o q.") from exc
    if not 0 <= selected_row_index < len(results_df):
        raise ValueError(f"La fila debe estar entre 0 y {len(results_df) - 1}.")
print("Matriz omitida." if selected_row_index is None else f"Fila seleccionada: {selected_row_index}")

## 8. Mostrar matriz de confusión

Se utilizan predicciones *out-of-fold* de validación cruzada, no predicciones sobre los mismos datos de entrenamiento.

In [ ]:
if selected_row_index is None:
    print("No se ha solicitado una matriz de confusión.")
else:
    selected_row = results_df.iloc[selected_row_index]
    model_name = selected_row["Algorithm"]
    params = selected_row["Parameters"]
    print(f"Modelo: {model_name}\nParámetros: {params}\nCV accuracy: {selected_row['CV accuracy']:.4f}\nOverfitting gap: {selected_row['Overfitting gap']:.4f}")
    selected_model = clone(selected_models[model_name][0])
    selected_model.set_params(**params)
    y_pred = cross_val_predict(selected_model, X, y, cv=cv, n_jobs=-1)
    classes = np.unique(y)
    matrix = confusion_matrix(y, y_pred, labels=classes)
    display_obj = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=classes)
    fig, ax = plt.subplots(figsize=(10, 8))
    display_obj.plot(ax=ax, xticks_rotation="vertical", values_format="d", colorbar=False)
    ax.set_title(f"Matriz de confusión - {model_name} (fila {selected_row_index})")
    plt.tight_layout()
    plt.show()